# J2C-TST-EN: reproducible evaluation

Evaluates the English baselines and any new system. The XLM-R classifier uses early stopping with patience 1 on validation accuracy. An external CSV only needs `row_id` and `generated_style`, one row per official test example. Add it to `EXTRA_SUBMISSIONS` or copy it into the English predictions directory.

In [1]:
from pathlib import Path
import os
import json
import subprocess
import sys

def find_evaluation_dir():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        for possible in (candidate, candidate / '6-evaluation'):
            if (possible / 'evaluate_metrics.py').is_file():
                return possible.resolve()
    raise FileNotFoundError('Cannot locate 6-evaluation/evaluate_metrics.py')

EVALUATION_DIR = find_evaluation_dir()
LANGUAGE = 'EN'
RUN_DIR = EVALUATION_DIR / 'J2C-TST-EN'
PREDICTIONS_DIR = EVALUATION_DIR.parent / '5-baselines' / 'J2C-TST-EN' / 'predictions'
CLASSIFIER_DIR = RUN_DIR / 'classifier' / 'early_stopping_p1'
RESULTS_DIR = RUN_DIR / 'results'
EXTRA_SUBMISSIONS = []  # e.g. [Path('/path/to/my_model_predictions.csv')]
ALLOW_PARTIAL = False  # keep False for comparable final results
INSTALL_DEPENDENCIES = True  # modern stack only; BELA uses BELA_PYTHON below
RUN_CLASSIFIER_TRAINING = True
RUN_MODERN_METRICS = True
RUN_BELA = True
METRIC_BATCH_SIZE = 32
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
BELA_PYTHON = EVALUATION_DIR / '.venv-bela' / 'bin' / 'python'
BELA_REPO = EVALUATION_DIR / 'external' / 'BELA'
def pick_atas_cache_base():
    explicit = os.environ.get('ATAS_CACHE_BASE')
    candidates = [explicit, '/tmp', f"/tmp/{os.environ.get('USER', '')}", f"/scratch/{os.environ.get('USER', '')}", os.environ.get('SCRATCH'), os.environ.get('LOCAL_SCRATCH'), os.environ.get('TMPDIR'), str(Path.home() / '.cache')]
    for value in candidates:
        if not value:
            continue
        candidate = Path(value).expanduser()
        try:
            candidate.mkdir(parents=True, exist_ok=True)
            probe = candidate / '.j2ctst_bela_write_test'
            probe.write_text('ok', encoding='utf-8')
            probe.unlink()
            return candidate.resolve()
        except OSError:
            continue
    raise RuntimeError('No writable directory found for BELA assets.')
ATAS_CACHE_BASE = pick_atas_cache_base()
default_bela_assets = ATAS_CACHE_BASE / 'BELA-models'
BELA_ASSETS_DIR = Path(os.environ.get('J2CTST_BELA_ASSETS_DIR', default_bela_assets)).expanduser().resolve()
print('Evaluation directory:', EVALUATION_DIR)
print('Prediction directory:', PREDICTIONS_DIR)
print('ATAS-compatible cache base:', ATAS_CACHE_BASE)
print('BELA assets directory:', BELA_ASSETS_DIR)

Evaluation directory: /home/user/J2C-TST/6-evaluation
Prediction directory: /home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions
ATAS-compatible cache base: /tmp
BELA assets directory: /tmp/BELA-models


In [2]:
if INSTALL_DEPENDENCIES:
    command = [sys.executable, '-m', 'pip', 'install', '-r', str(EVALUATION_DIR / 'requirements.txt')]
    print('>', subprocess.list2cmdline(command))
    subprocess.run(command, cwd=EVALUATION_DIR, check=True)
    if sys.platform.startswith('linux'):
        flash_env = os.environ.copy()
        flash_env.setdefault('MAX_JOBS', '2')
        flash_env.setdefault('NVCC_THREADS', '2')
        command = [sys.executable, '-m', 'pip', 'install', '--no-build-isolation', '-r', str(EVALUATION_DIR / 'requirements-flash-attn.txt')]
        print('>', subprocess.list2cmdline(command))
        subprocess.run(command, cwd=EVALUATION_DIR, env=flash_env, check=True)
    elif RUN_MODERN_METRICS:
        raise RuntimeError('The FlashAttention 2 profile requires a Linux CUDA environment.')
    if RUN_MODERN_METRICS:
        check = "import torch, flash_attn; assert torch.cuda.is_available(), 'CUDA unavailable'; assert torch.cuda.is_bf16_supported(), 'The GPU does not support BF16'; print(torch.cuda.get_device_name(0), 'BF16:', torch.cuda.is_bf16_supported(), 'flash-attn:', flash_attn.__version__)"
        subprocess.run([sys.executable, '-c', check], cwd=EVALUATION_DIR, check=True)
else:
    print('Modern dependency installation skipped; set INSTALL_DEPENDENCIES=True in the configuration cell for a fresh environment.')

> /opt/ice4hpc/rel-24.04/conda/bin/python -m pip install -r /home/user/J2C-TST/6-evaluation/requirements.txt
Defaulting to user installation because normal site-packages is not writeable
> /opt/ice4hpc/rel-24.04/conda/bin/python -m pip install --no-build-isolation -r /home/user/J2C-TST/6-evaluation/requirements-flash-attn.txt
Defaulting to user installation because normal site-packages is not writeable
NVIDIA L40S BF16: True flash-attn: 2.8.3


In [3]:
if RUN_BELA:
    command = [sys.executable, 'setup_bela.py', '--environment', BELA_PYTHON.parent.parent, '--repo', BELA_REPO, '--assets-dir', BELA_ASSETS_DIR]
    print('>', subprocess.list2cmdline([str(item) for item in command]))
    subprocess.run([str(item) for item in command], cwd=EVALUATION_DIR, check=True)
    if not BELA_PYTHON.is_file():
        raise FileNotFoundError(f'BELA setup completed without creating the expected interpreter: {BELA_PYTHON}')
else:
    print('BELA setup skipped because RUN_BELA=False.')

> /opt/ice4hpc/rel-24.04/conda/bin/python setup_bela.py --environment /home/user/J2C-TST/6-evaluation/.venv-bela --repo /home/user/J2C-TST/6-evaluation/external/BELA --assets-dir /tmp/BELA-models
Reusing BELA repository: /home/user/J2C-TST/6-evaluation/external/BELA

> /home/user/J2C-TST/6-evaluation/.venv-bela/bin/python -m pip install pip<24.1 setuptools==80.9.0


DEPRECATION: pytorch-lightning 1.7.7 has a non-standard dependency specifier torch>=1.9.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: /home/user/J2C-TST/6-evaluation/.venv-bela/bin/python -m pip install --upgrade pip



> /home/user/J2C-TST/6-evaluation/.venv-bela/bin/python -m pip install -r /home/user/J2C-TST/6-evaluation/requirements-bela.txt

> /home/user/J2C-TST/6-evaluation/.venv-bela/bin/python -m pip install -e /home/user/J2C-TST/6-evaluation/external/BELA


DEPRECATION: pytorch-lightning 1.7.7 has a non-standard dependency specifier torch>=1.9.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: /home/user/J2C-TST/6-evaluation/.venv-bela/bin/python -m pip install --upgrade pip


Obtaining file:///home/user/J2C-TST/6-evaluation/external/BELA
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'


DEPRECATION: pytorch-lightning 1.7.7 has a non-standard dependency specifier torch>=1.9.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063


  Attempting uninstall: bela
    Found existing installation: bela 0.1
    Uninstalling bela-0.1:
      Successfully uninstalled bela-0.1
  Running setup.py develop for bela

> /home/user/J2C-TST/6-evaluation/.venv-bela/bin/python -c "from importlib.metadata import version; assert version('setuptools') == '80.9.0', version('setuptools'); assert version('six') == '1.16.0', version('six'); import pkg_resources, six; from bela.datamodule.joint_el_datamodule import JointELDataModule; from bela.evaluation.model_eval import ModelEval; print('BELA runtime imports: OK; setuptools=' + version('setuptools') + '; six=' + version('six'))"



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: /home/user/J2C-TST/6-evaluation/.venv-bela/bin/python -m pip install --upgrade pip
<string>:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


BELA runtime imports: OK; setuptools=80.9.0; six=1.16.0
Reusing BELA asset: /tmp/BELA-models/model_e2e.ckpt
Reusing BELA asset: /tmp/BELA-models/index.txt
Reusing BELA asset: /tmp/BELA-models/embeddings.pt
BELA setup complete. Interpreter: /home/user/J2C-TST/6-evaluation/.venv-bela/bin/python


In [4]:
def run_script(arguments, python_executable=None):
    executable = python_executable or sys.executable
    command = [str(executable), *map(str, arguments)]
    print('\n>', subprocess.list2cmdline(command))
    process_env = os.environ.copy()
    if python_executable is not None and Path(executable) == Path(BELA_PYTHON):
        process_env.pop('PYTORCH_CUDA_ALLOC_CONF', None)
    return subprocess.run(command, cwd=EVALUATION_DIR, env=process_env, check=True)

classifier_args = ['train_style_classifier.py', '--language', LANGUAGE, '--output-dir', CLASSIFIER_DIR,
                   '--early-stopping-patience', '1', '--early-stopping-threshold', '0.0', '--resume']
if not RUN_CLASSIFIER_TRAINING:
    classifier_args.extend(['--max-train-pairs', '2', '--max-validation-pairs', '2', '--max-test-pairs', '2', '--dry-run'])
run_script(classifier_args)
if RUN_MODERN_METRICS and not RUN_CLASSIFIER_TRAINING and not (CLASSIFIER_DIR / 'config.json').is_file():
    raise FileNotFoundError(f'Train the style classifier first or set RUN_CLASSIFIER_TRAINING=True: {CLASSIFIER_DIR}')


> /opt/ice4hpc/rel-24.04/conda/bin/python train_style_classifier.py --language EN --output-dir /home/user/J2C-TST/6-evaluation/J2C-TST-EN/classifier/early_stopping_p1 --early-stopping-patience 1 --early-stopping-threshold 0.0 --resume


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map:   0%|          | 0/1946 [00:00<?, ? examples/s]

{
  "language": "EN",
  "corpus": "/home/user/J2C-TST/4-official_corpora/EN/J2C_TST_EN.csv",
  "model_id": "FacebookAI/xlm-roberta-base",
  "train_pairs": 973,
  "validation_pairs": 139,
  "test_pairs": 278,
  "labels": {
    "journalistic": 0,
    "conversational": 1
  },
  "max_length": 512,
  "epochs": 3.0,
  "early_stopping_patience": 1,
  "early_stopping_threshold": 0.0,
  "restore_callback_states_from_checkpoint": true,
  "learning_rate": 2e-05,
  "train_batch_size": 16,
  "eval_batch_size": 32,
  "gradient_accumulation_steps": 1,
  "seed": 42,
  "output_dir": "/home/user/J2C-TST/6-evaluation/J2C-TST-EN/classifier/early_stopping_p1"
}


Map: 100%|██████████| 556/556 [00:00<00:00, 1820.84 examples/s]
[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


No complete Trainer checkpoint found; starting classifier training from the beginning


...

{'eval_loss': 0.0002268835814902559, 'eval_accuracy': 1.0, 'eval_precision_conversational': 1.0, 'eval_recall_conversational': 1.0, 'eval_f1_conversational': 1.0, 'eval_runtime': 0.4952, 'eval_samples_per_second': 561.359, 'eval_steps_per_second': 18.173, 'epoch': 1.0}



 67%|██████▋   | 6/9 [00:00<00:00, 20.69it/s]
                                                 
 17%|█▋        | 3/18 [00:00<00:00, 26.65it/s]

{'eval_loss': 0.00025210619787685573, 'eval_accuracy': 1.0, 'eval_precision_conversational': 1.0, 'eval_recall_conversational': 1.0, 'eval_f1_conversational': 1.0, 'eval_runtime': 0.4911, 'eval_samples_per_second': 566.099, 'eval_steps_per_second': 18.327, 'epoch': 2.0}
{'train_runtime': 70.7307, 'train_samples_per_second': 82.538, 'train_steps_per_second': 5.175, 'train_loss': 0.07249755546694896, 'epoch': 2.0}


100%|██████████| 18/18 [00:00<00:00, 19.53it/s]


{
  "test_loss": 0.0002224994241259992,
  "test_accuracy": 1.0,
  "test_precision_conversational": 1.0,
  "test_recall_conversational": 1.0,
  "test_f1_conversational": 1.0,
  "test_runtime": 0.9881,
  "test_samples_per_second": 562.705,
  "test_steps_per_second": 18.217,
  "epoch": 2.0
}


In [5]:
prediction_files = list(PREDICTIONS_DIR.glob('*.csv')) if PREDICTIONS_DIR.exists() else []
for submission in EXTRA_SUBMISSIONS:
    submission = Path(submission).expanduser().resolve()
    if not submission.is_file():
        raise FileNotFoundError(f'Submission not found: {submission}')
    prediction_files.append(submission)
prediction_files = sorted(set(path.resolve() for path in prediction_files), key=str)
stems = [path.stem for path in prediction_files]
if len(stems) != len(set(stems)):
    raise ValueError('Submission filenames must have unique stems to avoid result collisions')
if not prediction_files:
    print('No submissions found. Expected CSV columns: row_id, generated_style')
prediction_files

[PosixPath('/home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_gemma_few_shot_gpu48_bf16_fa2_es1_gemma_b2.csv'),
 PosixPath('/home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_gemma_lora_gpu48_bf16_fa2_es1_gemma_b2.csv'),
 PosixPath('/home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_gemma_prefix_gpu48_bf16_fa2_es1_gemma_b2.csv'),
 PosixPath('/home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_llama_few_shot_gpu48_bf16_fa2_es1_b4.csv'),
 PosixPath('/home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_llama_lora_gpu48_bf16_fa2_es1_b4.csv'),
 PosixPath('/home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_llama_prefix_gpu48_bf16_fa2_es1_b4.csv')]

In [6]:
for prediction in prediction_files:
    result_dir = RESULTS_DIR / prediction.stem
    args = ['evaluate_metrics.py', '--language', LANGUAGE, '--predictions', prediction,
            '--style-classifier', CLASSIFIER_DIR, '--output-dir', result_dir,
            '--batch-size', METRIC_BATCH_SIZE, '--bf16',
            '--attention-implementation', 'flash_attention_2', '--resume']
    if ALLOW_PARTIAL:
        args.append('--allow-partial')
    if not RUN_MODERN_METRICS:
        args.append('--dry-run')
    run_script(args)


> /opt/ice4hpc/rel-24.04/conda/bin/python evaluate_metrics.py --language EN --predictions /home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_gemma_few_shot_gpu48_bf16_fa2_es1_gemma_b2.csv --style-classifier /home/user/J2C-TST/6-evaluation/J2C-TST-EN/classifier/early_stopping_p1 --output-dir /home/user/J2C-TST/6-evaluation/J2C-TST-EN/results/EN_gemma_few_shot_gpu48_bf16_fa2_es1_gemma_b2 --batch-size 32 --bf16 --attention-implementation flash_attention_2 --resume


The tokenizer you are loading from '/home/user/J2C-TST/6-evaluation/J2C-TST-EN/classifier/early_stopping_p1' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


{
  "language": "EN",
  "predictions": "/home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_gemma_few_shot_gpu48_bf16_fa2_es1_gemma_b2.csv",
  "system_id": "EN_gemma_few_shot_gpu48_bf16_fa2_es1_gemma_b2",
  "sha256": "64df9697be99ca5d3ef005f6754d1eb79ec30e96f17369948ae32417e48f50ef",
  "corpus": "/home/user/J2C-TST/4-official_corpora/EN/J2C_TST_EN.csv",
  "evaluated_rows": 278,
  "expected_test_rows": 278,
  "complete": true,
  "allow_partial": false
}
{
  "language": "EN",
  "system_id": "EN_gemma_few_shot_gpu48_bf16_fa2_es1_gemma_b2",
  "predictions": "/home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_gemma_few_shot_gpu48_bf16_fa2_es1_gemma_b2.csv",
  "examples": 278,
  "model_id": "google/gemma-3-1b-it",
  "strategy": "few_shot",
  "submission": {
    "system_id": "EN_gemma_few_shot_gpu48_bf16_fa2_es1_gemma_b2",
    "sha256": "64df9697be99ca5d3ef005f6754d1eb79ec30e96f17369948ae32417e48f50ef",
    "corpus": "/home/user/J2C-TST/4-official_corpora/EN/J2C_TST_EN.csv",
    "e

The tokenizer you are loading from '/home/user/J2C-TST/6-evaluation/J2C-TST-EN/classifier/early_stopping_p1' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


{
  "language": "EN",
  "predictions": "/home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_gemma_lora_gpu48_bf16_fa2_es1_gemma_b2.csv",
  "system_id": "EN_gemma_lora_gpu48_bf16_fa2_es1_gemma_b2",
  "sha256": "6ee1d7d9bf310e040177596d58885da22168fb95fc17029ad27ecaf660198575",
  "corpus": "/home/user/J2C-TST/4-official_corpora/EN/J2C_TST_EN.csv",
  "evaluated_rows": 278,
  "expected_test_rows": 278,
  "complete": true,
  "allow_partial": false
}
{
  "language": "EN",
  "system_id": "EN_gemma_lora_gpu48_bf16_fa2_es1_gemma_b2",
  "predictions": "/home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_gemma_lora_gpu48_bf16_fa2_es1_gemma_b2.csv",
  "examples": 278,
  "model_id": "google/gemma-3-1b-it",
  "strategy": "lora",
  "submission": {
    "system_id": "EN_gemma_lora_gpu48_bf16_fa2_es1_gemma_b2",
    "sha256": "6ee1d7d9bf310e040177596d58885da22168fb95fc17029ad27ecaf660198575",
    "corpus": "/home/user/J2C-TST/4-official_corpora/EN/J2C_TST_EN.csv",
    "evaluated_rows": 278,
   

...

{
  "language": "EN",
  "predictions": "/home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_llama_few_shot_gpu48_bf16_fa2_es1_b4.csv",
  "system_id": "EN_llama_few_shot_gpu48_bf16_fa2_es1_b4",
  "sha256": "0a3b5d1e56001346c222b476164bd558c5a1f11e66bd4b0f1ff84f13554338d1",
  "corpus": "/home/user/J2C-TST/4-official_corpora/EN/J2C_TST_EN.csv",
  "evaluated_rows": 278,
  "expected_test_rows": 278,
  "complete": true,
  "allow_partial": false
}
{
  "language": "EN",
  "system_id": "EN_llama_few_shot_gpu48_bf16_fa2_es1_b4",
  "predictions": "/home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_llama_few_shot_gpu48_bf16_fa2_es1_b4.csv",
  "examples": 278,
  "model_id": "meta-llama/Llama-3.2-1B",
  "strategy": "few_shot",
  "submission": {
    "system_id": "EN_llama_few_shot_gpu48_bf16_fa2_es1_b4",
    "sha256": "0a3b5d1e56001346c222b476164bd558c5a1f11e66bd4b0f1ff84f13554338d1",
    "corpus": "/home/user/J2C-TST/4-official_corpora/EN/J2C_TST_EN.csv",
    "evaluated_rows": 278,
    "e

The tokenizer you are loading from '/home/user/J2C-TST/6-evaluation/J2C-TST-EN/classifier/early_stopping_p1' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


{
  "language": "EN",
  "predictions": "/home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_llama_lora_gpu48_bf16_fa2_es1_b4.csv",
  "system_id": "EN_llama_lora_gpu48_bf16_fa2_es1_b4",
  "sha256": "39f00e20bbec232ed435227a3454f044b9ff0935ed4e1e49e63454bb1294d8f8",
  "corpus": "/home/user/J2C-TST/4-official_corpora/EN/J2C_TST_EN.csv",
  "evaluated_rows": 278,
  "expected_test_rows": 278,
  "complete": true,
  "allow_partial": false
}
{
  "language": "EN",
  "system_id": "EN_llama_lora_gpu48_bf16_fa2_es1_b4",
  "predictions": "/home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_llama_lora_gpu48_bf16_fa2_es1_b4.csv",
  "examples": 278,
  "model_id": "meta-llama/Llama-3.2-1B",
  "strategy": "lora",
  "submission": {
    "system_id": "EN_llama_lora_gpu48_bf16_fa2_es1_b4",
    "sha256": "39f00e20bbec232ed435227a3454f044b9ff0935ed4e1e49e63454bb1294d8f8",
    "corpus": "/home/user/J2C-TST/4-official_corpora/EN/J2C_TST_EN.csv",
    "evaluated_rows": 278,
    "expected_test_rows": 278,

The tokenizer you are loading from '/home/user/J2C-TST/6-evaluation/J2C-TST-EN/classifier/early_stopping_p1' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


{
  "language": "EN",
  "predictions": "/home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_llama_prefix_gpu48_bf16_fa2_es1_b4.csv",
  "system_id": "EN_llama_prefix_gpu48_bf16_fa2_es1_b4",
  "sha256": "05522eb53870bf92a0958945bd36f4fb2e989babf8987339f12ba6807161df9c",
  "corpus": "/home/user/J2C-TST/4-official_corpora/EN/J2C_TST_EN.csv",
  "evaluated_rows": 278,
  "expected_test_rows": 278,
  "complete": true,
  "allow_partial": false
}
{
  "language": "EN",
  "system_id": "EN_llama_prefix_gpu48_bf16_fa2_es1_b4",
  "predictions": "/home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_llama_prefix_gpu48_bf16_fa2_es1_b4.csv",
  "examples": 278,
  "model_id": "meta-llama/Llama-3.2-1B",
  "strategy": "prefix",
  "submission": {
    "system_id": "EN_llama_prefix_gpu48_bf16_fa2_es1_b4",
    "sha256": "05522eb53870bf92a0958945bd36f4fb2e989babf8987339f12ba6807161df9c",
    "corpus": "/home/user/J2C-TST/4-official_corpora/EN/J2C_TST_EN.csv",
    "evaluated_rows": 278,
    "expected_test

In [7]:
if RUN_BELA and (BELA_PYTHON is None or not Path(BELA_PYTHON).is_file()):
    raise FileNotFoundError(f'Create the dedicated BELA environment or set BELA_PYTHON: {BELA_PYTHON}')
for prediction in prediction_files:
    result_dir = RESULTS_DIR / prediction.stem / 'entities'
    args = ['entity_faithfulness.py', '--language', LANGUAGE, '--predictions', prediction,
            '--output-dir', result_dir, '--bela-repo', BELA_REPO,
            '--checkpoint', BELA_ASSETS_DIR / 'model_e2e.ckpt',
            '--embeddings', BELA_ASSETS_DIR / 'embeddings.pt',
            '--entity-index', BELA_ASSETS_DIR / 'index.txt',
            '--mention-threshold', '0.2', '--entity-threshold', '0.05', '--resume']
    if ALLOW_PARTIAL:
        args.append('--allow-partial')
    if not RUN_BELA:
        args.append('--dry-run')
    run_script(args, BELA_PYTHON if RUN_BELA else None)


> /home/user/J2C-TST/6-evaluation/.venv-bela/bin/python entity_faithfulness.py --language EN --predictions /home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_gemma_few_shot_gpu48_bf16_fa2_es1_gemma_b2.csv --output-dir /home/user/J2C-TST/6-evaluation/J2C-TST-EN/results/EN_gemma_few_shot_gpu48_bf16_fa2_es1_gemma_b2/entities --bela-repo /home/user/J2C-TST/6-evaluation/external/BELA --checkpoint /tmp/BELA-models/model_e2e.ckpt --embeddings /tmp/BELA-models/embeddings.pt --entity-index /tmp/BELA-models/index.txt --mention-threshold 0.2 --entity-threshold 0.05 --resume


/home/user/J2C-TST/6-evaluation/.venv-bela/lib/python3.9/site-packages/hydra/experimental/initialize.py:80: UserWarning: hydra.experimental.initialize_config_module() is no longer experimental. Use hydra.initialize_config_module().
  deprecation_warning(message=message)
/home/user/J2C-TST/6-evaluation/.venv-bela/lib/python3.9/site-packages/hydra/experimental/initialize.py:82: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  self.delegate = real_initialize_config_module(
/home/user/J2C-TST/6-evaluation/.venv-bela/lib/python3.9/site-packages/hydra/experimental/compose.py:25: UserWarning: hydra.experimental.compose() is no longer experimental. Use hydra.compose()
  deprecation_warning(message=message)
/home/user/J2C-TST/6-evaluation/.venv-bela/lib/python3.9/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'joint_el_mel_new': Defaults list is missing `_self_`. See http

{
  "language": "EN",
  "system_id": "EN_gemma_few_shot_gpu48_bf16_fa2_es1_gemma_b2",
  "submission_sha256": "64df9697be99ca5d3ef005f6754d1eb79ec30e96f17369948ae32417e48f50ef",
  "corpus": "/home/user/J2C-TST/4-official_corpora/EN/J2C_TST_EN.csv",
  "predictions": "/home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_gemma_few_shot_gpu48_bf16_fa2_es1_gemma_b2.csv",
  "rows": 278,
  "expected_test_rows": 278,
  "complete": true,
  "allow_partial": false,
  "reuse_linked_entities": null
}


100%|██████████| 1/1 [00:03<00:00,  3.44s/it]


{
  "examples": 278,
  "reference_comparison": {
    "precision_micro": 0.017871649065800164,
    "recall_micro": 0.5365853658536586,
    "f1_micro": 0.034591194968553465,
    "precision_macro": 0.06742452632475907,
    "recall_macro": 0.8968824940047962,
    "f1_macro": 0.07424046260466284,
    "true_positive_entities": 44,
    "generated_entities": 2462,
    "reference_entities": 82
  },
  "source_support": {
    "precision_micro": 0.8590576766856215,
    "precision_macro": 0.8689524829901148,
    "supported_generated_entities": 2115,
    "generated_entities": 2462,
    "source_entities": 11425
  },
  "entity_coverage": {
    "source_nonempty_examples": 278,
    "reference_nonempty_examples": 60,
    "generated_nonempty_examples": 264,
    "source_nonempty_rate": 1.0,
    "reference_nonempty_rate": 0.2158273381294964,
    "generated_nonempty_rate": 0.9496402877697842
  },
  "empty_set_convention": "precision=1 when generated set is empty; recall=1 when expected set is empty",
  "pred

...

{
  "examples": 278,
  "reference_comparison": {
    "precision_micro": 0.06918238993710692,
    "recall_micro": 0.13414634146341464,
    "f1_micro": 0.09128630705394192,
    "precision_macro": 0.7136690647482015,
    "recall_macro": 0.8177458033573142,
    "f1_macro": 0.5826310380267216,
    "true_positive_entities": 11,
    "generated_entities": 159,
    "reference_entities": 82
  },
  "source_support": {
    "precision_micro": 0.8679245283018868,
    "precision_macro": 0.9609027064063036,
    "supported_generated_entities": 138,
    "generated_entities": 159,
    "source_entities": 11425
  },
  "entity_coverage": {
    "source_nonempty_examples": 278,
    "reference_nonempty_examples": 60,
    "generated_nonempty_examples": 87,
    "source_nonempty_rate": 1.0,
    "reference_nonempty_rate": 0.2158273381294964,
    "generated_nonempty_rate": 0.3129496402877698
  },
  "empty_set_convention": "precision=1 when generated set is empty; recall=1 when expected set is empty",
  "predictions

/home/user/J2C-TST/6-evaluation/.venv-bela/lib/python3.9/site-packages/hydra/experimental/initialize.py:80: UserWarning: hydra.experimental.initialize_config_module() is no longer experimental. Use hydra.initialize_config_module().
  deprecation_warning(message=message)
/home/user/J2C-TST/6-evaluation/.venv-bela/lib/python3.9/site-packages/hydra/experimental/initialize.py:82: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  self.delegate = real_initialize_config_module(
/home/user/J2C-TST/6-evaluation/.venv-bela/lib/python3.9/site-packages/hydra/experimental/compose.py:25: UserWarning: hydra.experimental.compose() is no longer experimental. Use hydra.compose()
  deprecation_warning(message=message)
/home/user/J2C-TST/6-evaluation/.venv-bela/lib/python3.9/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'joint_el_mel_new': Defaults list is missing `_self_`. See http

{
  "language": "EN",
  "system_id": "EN_llama_prefix_gpu48_bf16_fa2_es1_b4",
  "submission_sha256": "05522eb53870bf92a0958945bd36f4fb2e989babf8987339f12ba6807161df9c",
  "corpus": "/home/user/J2C-TST/4-official_corpora/EN/J2C_TST_EN.csv",
  "predictions": "/home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_llama_prefix_gpu48_bf16_fa2_es1_b4.csv",
  "rows": 278,
  "expected_test_rows": 278,
  "complete": true,
  "allow_partial": false,
  "reuse_linked_entities": null
}


100%|██████████| 1/1 [00:03<00:00,  3.28s/it]


{
  "examples": 278,
  "reference_comparison": {
    "precision_micro": 0.0364963503649635,
    "recall_micro": 0.06097560975609756,
    "f1_micro": 0.045662100456621,
    "precision_macro": 0.7326139088729017,
    "recall_macro": 0.7955635491606714,
    "f1_macro": 0.5614594039054471,
    "true_positive_entities": 5,
    "generated_entities": 137,
    "reference_entities": 82
  },
  "source_support": {
    "precision_micro": 0.8394160583941606,
    "precision_macro": 0.9396282973621104,
    "supported_generated_entities": 115,
    "generated_entities": 137,
    "source_entities": 11425
  },
  "entity_coverage": {
    "source_nonempty_examples": 278,
    "reference_nonempty_examples": 60,
    "generated_nonempty_examples": 78,
    "source_nonempty_rate": 1.0,
    "reference_nonempty_rate": 0.2158273381294964,
    "generated_nonempty_rate": 0.2805755395683453
  },
  "empty_set_convention": "precision=1 when generated set is empty; recall=1 when expected set is empty",
  "predictions": "

In [8]:
summaries = []
for prediction in prediction_files:
    summary_path = RESULTS_DIR / prediction.stem / 'metrics_summary.json'
    if not summary_path.exists():
        continue
    item = json.loads(summary_path.read_text(encoding='utf-8'))
    entity_path = summary_path.parent / 'entities' / 'entity_metrics_summary.json'
    if entity_path.exists():
        item['entity_faithfulness'] = json.loads(entity_path.read_text(encoding='utf-8'))
    summaries.append(item)
if summaries:
    aggregate_path = RESULTS_DIR / 'all_results.json'
    aggregate_path.parent.mkdir(parents=True, exist_ok=True)
    aggregate_path.write_text(json.dumps(summaries, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Wrote', aggregate_path)
summaries

Wrote /home/user/J2C-TST/6-evaluation/J2C-TST-EN/results/all_results.json


[{'language': 'EN',
  'system_id': 'EN_gemma_few_shot_gpu48_bf16_fa2_es1_gemma_b2',
  'predictions': '/home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_gemma_few_shot_gpu48_bf16_fa2_es1_gemma_b2.csv',
  'examples': 278,
  'model_id': 'google/gemma-3-1b-it',
  'strategy': 'few_shot',
  'submission': {'system_id': 'EN_gemma_few_shot_gpu48_bf16_fa2_es1_gemma_b2',
   'sha256': '64df9697be99ca5d3ef005f6754d1eb79ec30e96f17369948ae32417e48f50ef',
   'corpus': '/home/user/J2C-TST/4-official_corpora/EN/J2C_TST_EN.csv',
   'evaluated_rows': 278,
   'expected_test_rows': 278,
   'complete': True,
   'allow_partial': False},
  'evaluation_config': {'bert_model': 'xlm-roberta-large',
   'perplexity_model': 'utter-project/EuroLLM-1.7B',
   'batch_size': 32,
   'perplexity_max_length': 4096,
   'attention_implementation': 'flash_attention_2',
   'bf16_required': True,
   'style_classifier': '/home/user/J2C-TST/6-evaluation/J2C-TST-EN/classifier/early_stopping_p1',
   'skip_style': False,
   's